In [0]:
%sql
ALTER TABLE customer360_dev.bronze.customers
ADD COLUMNS (
    _pipeline_run_id STRING
);

In [0]:
%sql
DESCRIBE TABLE customer360_dev.bronze.customers;

col_name,data_type,comment
customer_id,string,null
first_name,string,null
last_name,string,null
email,string,null
phone_number,string,null
date_of_birth,string,null
gender,string,null
signup_date,string,null
customer_status,string,null
loyalty_tier,string,null


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
import uuid

source_path = (
    "/Volumes/customer360_dev/staging/"
    "source_files/customers/incremental/"
)

target_table = "customer360_dev.bronze.customers"

run_id = str(uuid.uuid4())

print("Pipeline Run ID:", run_id)

Pipeline Run ID: 9fa5f9ad-c599-4137-a168-2ebcd838e396


In [0]:
customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone_number", StringType(), True),
    StructField("date_of_birth", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("signup_date", StringType(), True),
    StructField("customer_status", StringType(), True),
    StructField("loyalty_tier", StringType(), True),
    StructField("preferred_language", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("updated_at", StringType(), True)
])

In [0]:
incremental_df = (
    spark.read
    .option("header", True)
    .schema(customer_schema)
    .csv(source_path)
)


In [0]:

# ============================================================
# 4. Build Bronze Incremental DataFrame
# ============================================================

bronze_incremental_df = (
    incremental_df
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_source_system",
        F.lit("CRM")
    )
    .withColumn(
        "_load_type",
        F.lit("INCREMENTAL")
    )
    .withColumn(
        "_pipeline_run_id",
        F.lit(run_id)
    )
)


In [0]:

# ============================================================
# 5. Preview
# ============================================================

display(
    bronze_incremental_df.limit(20)
)


customer_id,first_name,last_name,email,phone_number,date_of_birth,gender,signup_date,customer_status,loyalty_tier,preferred_language,created_at,updated_at,_ingestion_timestamp,_source_file,_source_system,_load_type,_pipeline_run_id


In [0]:

# ============================================================
# 6. Count Incoming Records
# ============================================================

incoming_count = bronze_incremental_df.count()

print(
    f"Incoming customer incremental rows: "
    f"{incoming_count:,}"
)


Incoming customer incremental rows: 0


In [0]:

# ============================================================
# 7. Append to Bronze Only When Data Exists
# ============================================================

if incoming_count > 0:

    (
        bronze_incremental_df
        .write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(
            target_table
        )
    )

    print(
        f"Incremental batch loaded successfully. "
        f"run_id = {run_id}"
    )

else:

    print(
        "No incremental customer records found. "
        "Bronze write skipped."
    )

No incremental customer records found. Bronze write skipped.
